# [6.2] Gemma Scope Deep Dive - Solutions

Reference validation notebook for the section-local Gemma Scope implementation. This executes the visible tests against `solutions.py`, then checks the CPU notebook contract and the committed CUDA report highlights.

Expected CUDA highlights: the pinned Gemma Scope 2 1B-IT layer-13 residual JumpReLU SAE artifact loads on CUDA, tensor shapes match the locked config, a real encode/decode forward pass is finite, authenticated Gemma 3 residual activations pass semantic feature validation against random-feature and label-shuffle controls, feature validation and ablation controls pass, and peak VRAM stays inside budget.


In [ ]:
import json
import sys
from pathlib import Path

chapter = "chapter6_sparse_feature_methods"
section = "part2_gemma_scope_deep_dive"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part2_gemma_scope_deep_dive.tests as tests
from part2_gemma_scope_deep_dive import solutions


In [ ]:
tests.test_metadata_completeness_and_tag_selection(
    solutions.FeatureArtifactMetadata,
    solutions.TaggedFeatureSpec,
    solutions.metadata_is_complete,
    solutions.features_with_tag,
)
tests.test_feature_score_vector_reductions_match_reference(solutions.feature_score_vector)
tests.test_validate_feature_scores_beats_baseline_and_reports_means(
    solutions.validate_feature_scores,
    solutions.roc_auc_binary,
)
tests.test_base_instruction_delta_reports_signed_and_abs_change(
    solutions.base_instruction_feature_delta,
)
tests.test_ablation_control_requires_target_ablation_to_beat_random(
    solutions.ablation_control_report,
)
tests.test_steering_safety_report_checks_control_and_perplexity_guard(
    solutions.steering_safety_report,
)
tests.test_direct_logit_attribution_matches_selected_token_projection(
    solutions.direct_logit_attribution,
)
tests.test_notebook_contract(solutions.run_smoke_test)


In [ ]:
contract = solutions.run_smoke_test(cpu=True)
contract


In [ ]:
report = json.loads((section_dir / "verification_report.json").read_text())
gpu = report["metrics"]["gpu_test"]
assert gpu["gemma_scope_artifact_preflight_passed"], "Pinned Gemma Scope artifact preflight should pass."
assert gpu["gemma_scope_forward_passed"], "JumpReLU SAE encode/decode should execute on CUDA."
assert gpu["gemma_scope_repo_id"] == "google/gemma-scope-2-1b-it", "The artifact repo should remain pinned."
assert gpu["gemma_scope_revision"] == "b0fa29457c3601df0a70c48a15534c738d7c10e0", "The artifact revision should remain pinned."
assert gpu["gemma_scope_width"] == 16384, "Layer-13 small Gemma Scope width should be locked."
assert gpu["gemma_scope_d_model"] == 1152, "Gemma 2/3 1B residual stream width should be locked."
assert gpu["gemma_scope_w_enc_shape"] == [1152, 16384], "Encoder tensor shape should match the locked SAE config."
assert gpu["gemma_scope_w_dec_shape"] == [16384, 1152], "Decoder tensor shape should match the locked SAE config."
assert gpu["passes_baseline"], "Feature validation should beat the antipredictive baseline."
assert gpu["steering_passes_control"], "Steering should beat the random-feature control."
assert gpu["gemma_scope_semantic_feature_claimed"], "Report should claim semantic Gemma 3 activation validation only after authenticated real-activation controls pass."
assert gpu["gemma3_base_ready_for_real_activations"], "Current report should record authenticated Gemma 3 base weights for real activations."
assert gpu["gemma_scope_real_activation_preflight_passed"], "Real Gemma activation validation should pass."
assert gpu["gemma3_base_repo_listed"], "The gated base repo should still be discoverable."
assert gpu["within_vram_budget"], "CUDA verification should stay inside the configured VRAM budget."
{
    "feature_auc": gpu["feature_auc"],
    "baseline_auc": gpu["baseline_auc"],
    "gemma_scope_width": gpu["gemma_scope_width"],
    "gemma_scope_d_model": gpu["gemma_scope_d_model"],
    "gemma_scope_forward_passed": gpu["gemma_scope_forward_passed"],
    "gemma_scope_semantic_feature_claimed": gpu["gemma_scope_semantic_feature_claimed"],
    "peak_vram_gb": gpu["peak_vram_gb"],
}
